In [1]:
import numpy as np
import random
from statistics import mean, median, stdev
from copy import deepcopy
from Layers import ConcatenationLayer, Model_CPMP, Reduction, LayerExpandOutput, OutputMultiplication
from keras.models import Model, load_model
from CPMP import cpmp_ml
from CPMP.cpmp_ml import generate_data2, greedy_model, generate_random_layout
from CPMP.cpmp_ml import greedys, read_file, get_move, permutate_y
from CPMP.cpmp_ml import Layout, random_perturbate_layout, greedy, generate_y

In [2]:
def get_ann_state(layout: cpmp_ml.Layout) -> np.ndarray:
    S=len(layout.stacks) # Cantidad de stacks
    #matriz de stacks
    b = 2. * np.ones([S,layout.H + 1]) # Matriz normalizada
    for i,j in enumerate(layout.stacks):
        b[i][layout.H-len(j) + 1:] = [k/layout.total_elements for k in j]
        b[i][0] = layout.is_sorted_stack(i)
    b.shape=(S,(layout.H + 1))
    return b

In [3]:
def get_move(act, S=5,H=5):
  k=0
  for i in range(S):
    for j in range(S):
      if(i==j): continue
      if k==act: return (i,j)
      k+=1

In [4]:
def greedy_model(model, layouts, S= 5, H= 5, max_steps=10):
  from keras import backend as K
  costs = -np.ones(len(layouts))

  for steps in range(max_steps):
    x = []
    for i in range(len(layouts)):
      if layouts[i].unsorted_stacks==0: 
        if costs[i] ==-1: costs[i]=steps
        continue
      x.append(get_ann_state(layouts[i]))
    
    if len(x)==0: break
    actions = model.predict(np.stack(x), verbose=False)
    K.clear_session()
    k=0
    for i in range(len(layouts)):
      if costs[i] != -1: continue
      act = np.argmax(actions[k])
      move = get_move(act, S= S, H= H)
      layouts[i].move(move)
      k+=1
  return costs

In [5]:
def generate_data2(
    model,
    S=5,
    H=5,
    N=10,
    sample_size=1000,
    max_steps=20,
    batch_size=1000,
    perms_by_layout=20,
):
    x = []
    y = []

    while True:
        lays = []
        for i in range(batch_size):
            lays.append(generate_random_layout(S, H, N))
            # print ("Layout generado:", lays[i].stacks)

        lays0 = deepcopy(lays)
        costs = greedy_model(model, lays, S= S, H= H, max_steps=max_steps)

        # lays that cannot be solved by the model
        # lays0 = [lays0[i] for i in range(batch_size) if costs[i]==-1]
        # lays = [lays[i] for i in range(batch_size) if costs[i]==-1]
        # print("Costo obtenido por modelo:", len(lays))

        # for each lay we generate children clays
        clays = []
        for p in range(len(lays)):
            for i in range(S):
                for j in range(S):
                    if i == j:
                        continue
                    clay = deepcopy(lays0[p])
                    clay.move((i, j))
                    clays.append(clay)
            # print("len clays", len(clays))
        # print (f"clays generados {len(clays)}")
        # clays are solved

        ccosts = greedy_model(model, clays, S= S, H= H, max_steps=max_steps)
        # print("costs", ccosts)

        # f = lambda parent, k: (parent * (S*(S-1))) + k

        # Para cada padre
        for p in range(len(lays)):
            # print(lays[p].stacks)
            # print (ccosts[p*S*(S-1):(p+1)*S*(S-1)])
            A = []
            mincost = np.inf
            for c in range(p * (S * (S - 1)), (p + 1) * (S * (S - 1))):
                if ccosts[c] != -1 and ccosts[c] < mincost:
                    mincost = ccosts[c]

            if costs[p] != -1 and mincost >= costs[p]:
                continue

            for c in range(p * (S * (S - 1)), (p + 1) * (S * (S - 1))):
                if ccosts[c] != -1 and ccosts[c] == mincost:
                    A.append(1) 
                else:
                    A.append(0)

            if (
                sum(A) > 0
            ):  # otherwise no action was succesful, we simply discard the data
                for k in range(perms_by_layout):
                    enum_stacks = list(range(S))
                    perm = random.sample(enum_stacks, S)
                    lays0[p].permutate(perm)
                    A = permutate_y(A, S, perm)

                    x.append(get_ann_state(lays0[p]))
                    y.append(deepcopy(A))
                    if len(x) % 100 == 0: print(f'sample_size {len(x)}')
                    if len(x) == sample_size:
                        return x, y

In [6]:
def validate_model(model, S, H, N, verbose: bool = True, cvs_class=None):
    n = 1000

    lays = []
    if cvs_class is None:
        for i in range(n):
            lays.append(generate_random_layout(S,H,N))
    else:
        n=40
        for i in range(1,n+1):
            lay = read_file(f"benchmarks/CVS/{cvs_class}/data{cvs_class}-{i}.dat",5)
            lays.append(lay)

    lays1 = deepcopy(lays)
    costs1 = greedy_model(model, lays1, S= S, H= H, max_steps=N*2)
    costs2 = greedys(lays, max_steps= 80)

    valid_costs1 = [v for v in costs1 if v!=-1]
    valid_costs2 = [v for v in costs2 if v!=-1]

    results_model = len(valid_costs1) / n * 100.
    results_greedy = len(valid_costs2) / n * 100.

    if len(valid_costs1)>0:
        print(f"success ann model (%): {results_model}") 
        print(f"mean steps: {mean(valid_costs1)}")
        print(f"median steps: {median(valid_costs1)}")
        #print(f"stdesv steps: {stdev(valid_costs1)}")
        print(f"min steps: {min(valid_costs1)}")
        print(f"max steps: {max(valid_costs1)}")
        print('')
    if len(valid_costs2)==0:
        print("success heuristic (%):", results_greedy)
    else:
        print("success heuristic (%):", results_greedy, mean(valid_costs2))
        print(f"mean steps: {mean(valid_costs2)}")
        print(f"median steps: {median(valid_costs2)}")
        #print(f"stdesv steps: {stdev(valid_costs2)}")
        print(f"min steps: {min(valid_costs2)}")
        print(f"max steps: {max(valid_costs2)}")
        print('')

    return results_model, results_greedy

In [7]:
def reinforcement_training(model: Model, S: int = 5, H: int = 5, MPC: int = 15, 
                           sample_size: int = 100000, iter: int = 5, max_steps: int = 30, epochs: int = 5, 
                           batch_size: int = 20, verbose: bool = True, perms_by_layout: int = 1) -> None:
    for i in range(iter):
        if verbose: print(f"Step {i + 1}")

        data, labels = generate_data2(model= model, S= S, H= H, N= MPC, sample_size= sample_size, max_steps= max_steps
                                      , batch_size= batch_size, perms_by_layout= perms_by_layout)

        data = np.stack(data)
        labels = np.stack(labels)

        model.fit(data, labels, epochs= epochs, verbose= verbose)
        results_model, results_greedy = validate_model(model, S, H, MPC, verbose= verbose)

        del data, labels

        if verbose: print('')
        if results_model > 96.0: break

In [8]:
def load_cpmp_model(name: str) -> Model:
    custom_objects = {'Model_CPMP': Model_CPMP, 
                      'OutputMultiplication': OutputMultiplication,
                      'LayerExpandOutput': LayerExpandOutput,
                      'ConcatenationLayer': ConcatenationLayer,
                      'Reduction': Reduction}
    
    model = load_model(name, custom_objects= custom_objects) 

    return model

In [9]:
def eval_action(state, action, params):
    s_o, s_d = action
    g_s_d = state.gvalue(s_d)
    g_s_o = state.gvalue(s_o)
    c = state.stacks[s_o][-1]

    if state.is_BG_action(action):
        diff = g_s_d - g_s_o
        if state.reduced_stack == -1:
            return 100 - diff

    if state.reduced_stack==s_o or state.reduced_stack==-1:
        top_d = state.gvalue(s_d)

        if state.is_sorted_stack(s_d) and c <= top_d:  # xg
            eval_dest_stack = -top_d  # minimum difference between c and top_d is preferred
        elif not state.is_sorted_stack(s_d) and c >= top_d:  # xb
            eval_dest_stack = -10**params[0] + top_d  # minimum difference between c and top_d is preferred
        elif state.is_sorted_stack(s_d):  # xb
            eval_dest_stack = -10**params[1] - len(state.stacks[s_d])  # - top_d
        else:
            eval_dest_stack = -10**params[2] - 10**params[3]*len(state.stacks[s_d]) - top_d

        # Factor in remaining containers in the destination stack
        if len(state.stacks[s_d]) > 1:
            next_container = state.stacks[s_d][-2]
            if next_container > c:
                eval_dest_stack -= 10**params[4]  # Penalize this action


        stack_len_multiplier = 1 + len(state.stacks[s_o]) / state.H  # Factor in stack length dynamically
        return stack_len_multiplier * eval_dest_stack

    return float("-inf")

def greedy(state, basic=True, params=[2.0, 2.0, 4, 2.1, 2], max_steps=20) -> int:
    steps = 0
    while state.unsorted_stacks>0 and steps < max_steps:
        actions = state.get_actions()

        best_ev = float("-inf"); best_action=None
        for action in actions:
            ev = eval_action(state, action, params)
            if ev > best_ev:
              best_ev=ev
              best_action=action

        if best_action is not None:
            #print(best_ev,best_action)
            state.move(best_action)
            #print(state.stacks)
        else:
            return -1
        steps +=1

    if state.unsorted_stacks==0:
        return steps
    return -1

def get_actions(self):
    actions =[]
    for i in range(len(self.stacks)):
        for j in range(len(self.stacks)):
            if i!=j and len(self.stacks[i]) > 0 and len(self.stacks[j]) < self.H:
                    actions.append((i,j))
    return actions

def is_BG_action(self, action):
    s_o = action[0]; s_d = action[1]
    if (self.is_sorted_stack(s_o)==False
    and self.is_sorted_stack(s_d)==True
    and self.gvalue(s_o) <= self.gvalue(s_d)):
      return True

    else: return False

def greedys(layouts, max_steps= 20):
  costs = -np.ones(len(layouts))
  for k in range(len(layouts)):
    steps = greedy(layouts[k], max_steps= max_steps)
    costs[k]=steps
  return costs

In [10]:
cpmp_ml.generate_data2 = generate_data2
cpmp_ml.greedy_model = greedy_model
cpmp_ml.get_ann_state = get_ann_state
cpmp_ml.get_move = get_move
cpmp_ml.greedys = greedys

In [11]:
#overwriting greedy v2

cpmp_ml.greedy=greedy
Layout.get_actions=get_actions
Layout.is_BG_action=is_BG_action

In [12]:
model = load_cpmp_model('models/model_cpmp_10x7_v2.keras')

In [15]:
validate_model(model, S= 10, H= 7, N= 50, verbose= True)

success heuristic (%): 98.0 55.662244897959184
mean steps: 55.662244897959184
median steps: 56.0
min steps: 35.0
max steps: 75.0



(0.0, 98.0)

In [14]:
reinforcement_training(model= model, S= 10, H= 7, MPC= 28, sample_size= 100000, 
                       iter= 2, batch_size= 55, max_steps= 28 * 2)

Step 1
sample_size 100
sample_size 200
sample_size 300
sample_size 400
sample_size 500
sample_size 600
sample_size 700
sample_size 800
sample_size 900
sample_size 1000
sample_size 1100
sample_size 1200
sample_size 1300
sample_size 1400
sample_size 1500
sample_size 1600
sample_size 1700
sample_size 1800
sample_size 1900
sample_size 2000
sample_size 2100
sample_size 2200
sample_size 2300
sample_size 2400
sample_size 2500
sample_size 2600
sample_size 2700
sample_size 2800
sample_size 2900
sample_size 3000
sample_size 3100
sample_size 3200
sample_size 3300
sample_size 3400
sample_size 3500
sample_size 3600
sample_size 3700
sample_size 3800
sample_size 3900
sample_size 4000
sample_size 4100
sample_size 4200
sample_size 4300
sample_size 4400
sample_size 4500
sample_size 4600
sample_size 4700
sample_size 4800
sample_size 4900
sample_size 5000
sample_size 5100
sample_size 5200
sample_size 5300
sample_size 5400
sample_size 5500
sample_size 5600
sample_size 5700
sample_size 5800
sample_size 5900

In [ ]:
model.save('models/model_cpmp_10x7.keras')